# 🧪 W4-D7 总复习实验：一道题，四种解法，谁答得上谁答不上？

> 配套阅读：`ima/第4周-Day7-第四周总复习.md`（全周知识全景图、20 题综合测试、术语总表在那边）
>
> 这不是新知识，是把整周的四个角色拉到同一张考卷前：
> **闭卷 LLM / 向量 RAG / 混合 RAG（BM25+RRF）/ GraphRAG**，6 道覆盖不同题型的问题，
> 谁能答、谁不能答、为什么——全部用代码实测，最后画成一张命中矩阵。
>
> 实验环境：纯 Python/numpy，W3-D1~D5 各实验的浓缩版。

## 实验 1：迷你 benchmark —— 6 道题 × 4 种方法

题型刻意错开：时效事实（Q1）、精确符号（Q2）、语义泛化（Q3）、多跳关系（Q4/Q5）、语料外常识（Q6）。
每种方法用当天教过的最小实现：词典向量（D3）、BM25+RRF（D3）、三元组 BFS（D4）、闭卷=过时内部知识（D1）。

In [ ]:
import math
from collections import defaultdict, deque
import numpy as np

# ---------- 语料与知识图谱（浓缩自 D3/D4） ----------
DOCS = {
    "D1": "杨枝甘露：芒果与西柚果肉，椰浆打底，冰镇出品，夏季限定，售价20元。",
    "D2": "芒果双皮奶，产品编号SKU-207，双份奶皮，售价18元。",
    "D4": "芋泥波波冰：芋头现蒸捣泥，黑糖波波铺顶，冰沙绵密，售价17元。",
    "D6": "西瓜冰：纯西瓜果肉冰沙，无奶配方，夏天人气王，售价16元。",
    "D7": "桂花酸梅汤：古法熬煮，生津解腻，开胃消食，售价10元。",
}
TRIPLES = [
    ("杨枝甘露", "含原料", "炼乳"), ("杨枝甘露", "含原料", "芒果"),
    ("芒果双皮奶", "含原料", "芒果"), ("芒果双皮奶", "含原料", "水牛奶"),
    ("炼乳", "属于", "乳制品"), ("水牛奶", "属于", "乳制品"),
    ("芒果", "含过敏原", "漆酚类蛋白"), ("乳制品", "含过敏原", "乳糖蛋白"),
]

# ---------- 方法1：闭卷 LLM（知识截止于旧菜单） ----------
CLOSED = {"新菜单价格": "旧菜单是 12 元", "营业时间": "早十点到晚十点"}
def closed_book(q):
    if "营业" in q or "开门" in q or "几点" in q:
        return CLOSED["营业时间"], True
    return CLOSED["新菜单价格"] + "（拿旧知识硬答）", False

# ---------- 方法2：词典向量（模拟 Embedding） ----------
TOPICS = [["冰镇", "冰沙", "夏季", "夏天", "冰爽", "解暑", "冰"],
          ["芒果", "西瓜", "西柚", "果肉", "水果"],
          ["奶", "牛奶", "双皮奶", "椰浆", "乳"], ["解腻", "开胃", "生津", "酸梅"]]
DOC_EMB = {k: np.array([sum(t.count(w) for w in kws) for kws in TOPICS]) for k, t in DOCS.items()}
def vec_search(q, k=3):
    v = np.array([sum(q.count(w) for w in kws) for kws in TOPICS], dtype=float)
    if np.linalg.norm(v) < 1e-9:
        return []
    vn = v / np.linalg.norm(v)
    sims = {d: float(vn @ (e / np.linalg.norm(e))) if np.linalg.norm(e) > 0 else 0.0
            for d, e in DOC_EMB.items()}
    return sorted(sims.items(), key=lambda kv: -kv[1])[:k]

# ---------- 方法3：BM25 + RRF（混合检索） ----------
def bigrams(t):
    t = "".join(c for c in t if c not in "，。：、！？")
    return [t[i:i+2] for i in range(len(t) - 1)]
DOC_TOK = {k: bigrams(t) for k, t in DOCS.items()}
DF = {}
for toks in DOC_TOK.values():
    for g in set(toks):
        DF[g] = DF.get(g, 0) + 1
AVGDL = np.mean([len(t) for t in DOC_TOK.values()])
def bm25(q, k=3):
    qg = [g for g in bigrams(q) if DF.get(g)]
    sc = {}
    for d, toks in DOC_TOK.items():
        s = 0.0
        for g in qg:
            tf = toks.count(g)
            if tf:
                idf = math.log((len(DOCS) - DF[g] + 0.5) / (DF[g] + 0.5) + 1)
                s += idf * tf * 2.5 / (tf + 1.5 * (0.25 + 0.75 * len(toks) / AVGDL))
        sc[d] = s
    return sorted(sc.items(), key=lambda kv: -kv[1])[:k]
def rrf(rankings, k=60):
    s = defaultdict(float)
    for r in rankings:
        for i, (d, _) in enumerate(r, 1):
            s[d] += 1 / (k + i)
    return sorted(s.items(), key=lambda kv: -kv[1])
def hybrid(q, k=3):
    return rrf([vec_search(q, 5), bm25(q, 5)])[:k]

# ---------- 方法4：GraphRAG（三元组 BFS） ----------
adj = defaultdict(list)
for h, r, t in TRIPLES:
    adj[h].append((r, t))
def k_hop(start, hops):
    seen, frontier = {start}, deque([start])
    for _ in range(hops):
        nxt = set()
        for e in frontier:
            for rel, nb in adj[e]:
                if nb not in seen:
                    nxt.add(nb)
        seen |= nxt
        frontier = deque(nxt)
    return seen - {start}

# ---------- 出卷与判分 ----------
QUESTIONS = [
    ("Q1 时效事实：芋泥波波冰售价多少", "价格在语料里(D4)", lambda q: any("17元" in DOCS[d] for d, _ in vec_search(q, 3)),
     lambda q: any("17元" in DOCS[d] for d, _ in hybrid(q, 3)), False),
    ("Q2 精确符号：SKU-207 对应什么产品", "罕见编号(D2)", lambda q: any(d == "D2" for d, _ in vec_search(q, 3)),
     lambda q: any(d == "D2" for d, _ in hybrid(q, 3)), False),
    ("Q3 语义泛化：夏天想喝冰爽的水果甜品", "冰品家族(D1/D4/D6)", lambda q: vec_search(q, 1) != [] and vec_search(q, 1)[0][0] in {"D1", "D4", "D6"},
     lambda q: hybrid(q, 1) != [] and hybrid(q, 1)[0][0] in {"D1", "D4", "D6"}, False),
    ("Q4 多跳关系：杨枝甘露涉及哪些过敏原", "2跳可达过敏原", lambda q: False, lambda q: False, True),
    ("Q5 共享原料：芒果双皮奶和杨枝甘露共用什么", "交集=芒果", lambda q: False, lambda q: False, True),
    ("Q6 语料外常识：你们店几点开门", "语料里没有→兜底", lambda q: False, lambda q: False, False),
]

def closed_hit(qtext, is_q6):
    return closed_book(qtext)[1] if is_q6 else False

MATRIX = {}
for label, note, vfn, hfn, graph_q in QUESTIONS:
    qtext = label.split("：", 1)[1]
    MATRIX[label] = {
        "闭卷LLM": closed_hit(qtext, "开门" in qtext or "几点" in qtext),
        "向量RAG": bool(vfn(qtext)),
        "混合RAG": bool(hfn(qtext)),
        "GraphRAG": graph_q and (("杨枝甘露" in qtext and len(k_hop("杨枝甘露", 2) & {"漆酚类蛋白", "乳糖蛋白"}) > 0)
                                  or ("共用" in qtext and len(k_hop("芒果双皮奶", 1) & k_hop("杨枝甘露", 1)) > 0)),
    }

hdr = f"{'问题':<22}" + "".join(f"{m:>10}" for m in ["闭卷LLM", "向量RAG", "混合RAG", "GraphRAG"])
print(hdr); print("-" * len(hdr))
for label, row in MATRIX.items():
    print(f"{label:<24}" + "".join(f"{'✓' if row[m] else '✗':>10}" for m in ["闭卷LLM", "向量RAG", "混合RAG", "GraphRAG"]))
print("\n判分口径：闭卷=是否答对；RAG系=正确chunk是否进Top-3；GraphRAG=图上是否可达答案")

## 实验 2：命中矩阵可视化 + 各方法真实耗时

把实验 1 的矩阵画成热力图，并用 `time.perf_counter` 实测四种方法的单次查询耗时
（各跑 500 次取均值）——生产选型永远是"能力 × 成本"的二维决策。

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

methods = ["闭卷LLM", "向量RAG", "混合RAG", "GraphRAG"]
qs = list(MATRIX.keys())
M = np.array([[1 if MATRIX[q][m] else 0 for m in methods] for q in qs])

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.imshow(M, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(methods)), methods)
ax.set_yticks(range(len(qs)), [q.split("：")[0] for q in qs])
for i in range(len(qs)):
    for j in range(len(methods)):
        ax.text(j, i, "√" if M[i, j] else "×", ha="center", va="center", fontsize=16)
ax.set_title("W4 全周知识浓缩：6 道题 × 4 种方法的命中矩阵")
plt.tight_layout(); plt.show()

# ---- 真实耗时（每方法 500 次取均值） ----
def bench(fn, n=500):
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - t0) / n * 1000

q_bench = "夏天想喝冰爽的水果甜品"   # 同一查询，公平计时
lat = {
    "闭卷LLM": bench(lambda: closed_book("芋泥波波冰售价多少")),
    "向量RAG": bench(lambda: vec_search(q_bench, 3)),
    "混合RAG": bench(lambda: hybrid(q_bench, 3)),
    "GraphRAG": bench(lambda: k_hop("杨枝甘露", 2)),
}
for m, ms in lat.items():
    print(f"{m:<10} 单次查询 ≈ {ms*1000:.1f} µs")
print("\n复盘：混合RAG=两路检索+融合，最贵但覆盖面最广；图遍历在这个小图上极快，")
print("     且随边数线性增长——成本结构和向量检索完全不同（这决定了两者的适用场景）。")

## 实验 3：随堂自测 —— 抽 6 张术语卡，先自己答再看答案

术语卡来自全周 md 的术语表浓缩。程序随机抽卡、打印"题面"，你心里作答后再看"答案"。

In [ ]:
import numpy as np

CARDS = {
    "RAG": "检索增强生成：生成前先检索外部知识，用上下文约束模型输出，解决知识截止与幻觉。",
    "chunk（分块）": "文档切割后的检索单元；大小要对齐'一个完整事实'，边界会吞掉跨界信息（W4-D1/D5）。",
    "余弦相似度": "只看向量方向夹角、不管幅度；归一化后与欧氏距离等价：‖a-b‖²=2(1-cos)（W4-D2）。",
    "TF-IDF": "词频×逆文档频率：常见词权重低、罕见词权重高；本系列用它模拟 Embedding 的骨架。",
    "BM25": "关键词检索打分公式：词频饱和(k1) + 文档长度归一(b) + IDF；精确匹配罕见符号的最强者（W4-D3）。",
    "RRF": "倒数排名融合：score=Σ1/(k+rank)，k常取60；只比名次不比分数，天然免疫量纲问题（W4-D3）。",
    "Multi-Query": "查询改写：一个查询生成多个变体分别检索再融合，跨词汇鸿沟，代价是N倍检索（W4-D3）。",
    "重排序（Rerank）": "粗排召回候选后用更精细的模型重排；只能排不能捞——粗排深度决定召回上限（W4-D3）。",
    "知识三元组": "(头实体, 关系, 尾实体)，知识图谱最小单元；BFS多跳遍历回答关系型问题（W4-D4）。",
    "GraphRAG": "把检索变成图上的关系遍历，擅长多跳/溯源/排除类问题，答案路径可解释（W4-D4）。",
    "lost in the middle": "LLM对长上下文中间位置注意力弱（U形）；K调大召回升、有效利用率反降（W4-D5）。",
    "混合检索（Hybrid）": "向量(语义泛化) + BM25(精确符号) 双路互补，用RRF融合；解决单一检索器的结构盲区（W4-D3）。",
}

rng = np.random.default_rng()
for key in rng.choice(sorted(CARDS), size=6, replace=False):
    print(f"❓ {key}")
    print(f"   → {CARDS[key]}\n")

# 附：RRF 公式手算自检（拿两张榜单验证你记的公式对不对）
r1, r2 = [("D1", 0), ("D2", 0)], [("D2", 0), ("D1", 0)]
manual = {"D1": 1/61 + 1/62, "D2": 1/62 + 1/61}
fused = dict(rrf([r1, r2]))
assert abs(fused["D1"] - manual["D1"]) < 1e-12
print("RRF 手算自检通过：两榜第一的文档得 1/61+1/62 =", round(fused['D1'], 5))

## 结论：全周一张表

| 方法 | 强项 | 结构性短板 | 实验证据 |
|---|---|---|---|
| 闭卷 LLM | 常识/推理 | 知识截止、幻觉 | 只有 Q6 ✓ |
| 向量 RAG | 语义泛化、事实命中 | 精确符号、多跳关系 | Q1/Q3 ✓，Q2/Q4/Q5 ✗ |
| 混合 RAG | 语义+符号全覆盖 | 仍然不懂"关系" | Q1/Q2/Q3 ✓，Q4/Q5 ✗ |
| GraphRAG | 多跳关系、可解释 | 时效事实仍靠文本 | Q4/Q5 ✓，Q1 ✗ |

**没有银弹：生产系统是"混合 RAG + 图谱 + 兜底策略"的乐高**——这正是 W5 之后 Agent 编排要解决的问题。

→ 系统复习：`ima/第4周-Day7-第四周总复习.md`（知识全景图、20 题测试、术语总表）